# Practice 02 — Math & Optimization

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theDocWho/lbv-notebooks/blob/main/ai-ml/02-math-optimization.ipynb) [![Open in Kaggle](https://img.shields.io/badge/Kaggle-open-20BEFF?logo=kaggle&logoColor=white)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/theDocWho/lbv-notebooks/main/ai-ml/02-math-optimization.ipynb)

Companion drills for **Phase 2** of [Learn by Visualization](https://learn-by-visuallization.org/illustrated/index.html) —
pairs with [dot product](https://learn-by-visuallization.org/illustrated/2-math/dot-product.html),
[matrix transformations](https://learn-by-visuallization.org/illustrated/2-math/matrix-transformation.html),
[eigenvectors](https://learn-by-visuallization.org/illustrated/2-math/eigenvectors.html) and the
[optimizer race](https://learn-by-visuallization.org/illustrated/2-math/optimizer-comparison.html) you can scrub step-by-step.

**How this works:** each exercise is a `# TODO` scaffold followed by a self-check cell. Fill in the TODO, re-run both cells, and turn the ❌ into a ✅. No answers are hidden in the notebook — the checks themselves tell you what "correct" means. Runs on local Jupyter, Google Colab, or Kaggle (free, no setup).

In [ ]:
NEEDED = [("numpy", "numpy")]

# ---- setup: run me first ----
import importlib.util, subprocess, sys
for pkg, pipname in NEEDED:
    if importlib.util.find_spec(pkg) is None:
        print(f"installing {pipname} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pipname])

RESULTS = {}
def check(name, fn, hint=""):
    """Self-check: PASS/FAIL with a hint. Never raises -- rerun the cell after editing your answer."""
    try:
        ok = bool(fn())
    except Exception as e:
        ok, hint = False, f"{hint} (your code raised {type(e).__name__}: {e})"
    RESULTS[name] = ok
    print(("\u2705 PASS" if ok else "\u274c FAIL") + f" \u2014 {name}" + ("" if ok else f"\n   hint: {hint}"))

import numpy as np
rng = np.random.default_rng(0)


### Exercise 1 — cosine similarity from scratch
Implement `cosine(a, b)` using **only** `np.dot` and `np.linalg.norm` — the similarity behind
every embedding search. Return 0.0 if either vector is all zeros.

*Hint: dot(a, b) / (‖a‖ ‖b‖).*

In [ ]:
def cosine(a, b):
    # TODO
    ...


In [ ]:
check("parallel vectors -> 1",
      lambda: abs(cosine(np.array([1., 2.]), np.array([2., 4.])) - 1) < 1e-9,
      "same direction = cosine 1")
check("orthogonal vectors -> 0",
      lambda: abs(cosine(np.array([1., 0.]), np.array([0., 3.]))) < 1e-9,
      "perpendicular = cosine 0")
check("opposite vectors -> -1",
      lambda: abs(cosine(np.array([1., 1.]), np.array([-2., -2.])) + 1) < 1e-9,
      "opposite direction = -1")
check("zero vector -> 0.0",
      lambda: cosine(np.zeros(3), np.array([1., 2., 3.])) == 0.0,
      "guard the division")

### Exercise 2 — a matrix IS a transformation
Build `R`: the 2×2 matrix that rotates the plane **90° counter-clockwise**, and apply it to the
points `pts` (shape n×2) to get `rot`.

*Hint: R = [[0, -1], [1, 0]]; rotating row-vectors is `pts @ R.T`.*

In [ ]:
pts = np.array([[1., 0.], [0., 1.], [2., 3.]])

R   = None  # TODO: 90-degree CCW rotation matrix
rot = None  # TODO: pts rotated


In [ ]:
check("R sends x-axis to y-axis",
      lambda: R is not None and np.allclose(R @ np.array([1., 0.]), [0., 1.]),
      "column view: R @ e1 is where e1 lands")
check("all points rotated correctly",
      lambda: rot is not None and np.allclose(rot, np.array([[0., 1.], [-1., 0.], [-3., 2.]])),
      "(x, y) -> (-y, x); with row vectors use pts @ R.T")

### Exercise 3 — gradient descent on the ravine
The explainer's loss: `f(x, y) = 0.5·(x² + 20·y²)`, gradient `(x, 20y)`. Implement
`gd(lr, steps)` from start `(-4.5, 3.6)` returning the final point as an array.
Then watch the **stability cliff**: the steep direction obeys `|1 − 20·lr| < 1` ⇒ lr < 0.1.

*Hint: loop `p = p - lr * grad(p)`.*

In [ ]:
def grad(p):
    return np.array([p[0], 20 * p[1]])

def loss(p):
    return 0.5 * (p[0]**2 + 20 * p[1]**2)

def gd(lr, steps):
    p = np.array([-4.5, 3.6])
    # TODO: take `steps` gradient steps
    ...


In [ ]:
check("lr=0.08 converges (loss < 1e-3 after 200 steps)",
      lambda: loss(gd(0.08, 200)) < 1e-3,
      "p = p - lr * grad(p), repeated")
check("lr=0.12 diverges along y (loss explodes)",
      lambda: loss(gd(0.12, 60)) > 1e3,
      "the y-multiplier is 1 - 20*0.12 = -1.4: each step GROWS y by 40%")
check("lr=0.01 is stable but slow (still > 1e-3 after 60 steps)",
      lambda: 1e-3 < loss(gd(0.01, 60)) < loss(np.array([-4.5, 3.6])),
      "tiny steps converge geologically -- compute is money")

### Exercise 4 — momentum beats the ravine
Implement `gd_momentum(lr, steps, beta=0.9)`: velocity `v = beta·v − lr·grad`, then `p = p + v`.
The check races both at lr = 0.02 for 150 steps — momentum should land **at least 100× lower**.

*Hint: initialize `v = np.zeros(2)`; scrub the [optimizer race](https://learn-by-visuallization.org/illustrated/2-math/optimizer-comparison.html) to see why it wins.*

In [ ]:
def gd_momentum(lr, steps, beta=0.9):
    p, v = np.array([-4.5, 3.6]), np.zeros(2)
    # TODO
    ...


In [ ]:
check("momentum lands >=100x lower than plain GD (lr=0.02, 150 steps)",
      lambda: loss(gd_momentum(0.02, 150)) * 100 <= loss(gd(0.02, 150)),
      "v = beta*v - lr*grad(p); p = p + v -- the velocity cancels the zig-zag")

### Exercise 5 — the top eigenvector is the data's main street
`cloud` is correlated 2-D data stretched along the y = x diagonal. Compute the covariance
matrix's **top eigenvector** `v1` (unit length). It should align with `[1, 1]/√2` — this is PCA's
first component, found by hand.

*Hint: `C = np.cov(cloud.T)`, then `np.linalg.eigh(C)` (eigh sorts ascending — take the LAST column).*

In [ ]:
base = rng.normal(size=(300,))
cloud = np.column_stack([base + 0.3 * rng.normal(size=300),
                         base + 0.3 * rng.normal(size=300)])

v1 = None  # TODO: unit top eigenvector of the covariance matrix


In [ ]:
check("v1 is unit length",
      lambda: v1 is not None and abs(np.linalg.norm(v1) - 1) < 1e-6,
      "eigh returns unit eigenvectors already")
check("v1 aligns with the diagonal (|cos| > 0.98)",
      lambda: abs(float(v1 @ (np.ones(2) / np.sqrt(2)))) > 0.98,
      "eigh's eigenvalues are ASCENDING: the top eigenvector is eigvecs[:, -1]")

### Stretch goal — implement Adam
Add `gd_adam(lr, steps)` (EMAs of grad and grad², bias correction, per-coordinate step) and verify
it survives learning rates that blow plain GD up — then re-watch the race with fresh eyes.

In [ ]:
# stretch — your playground


In [ ]:
# ---- self-score ----
passed, total = sum(RESULTS.values()), len(RESULTS)
print(f"Self-score: {passed}/{total} checks passing")
print("\U0001f389 All green \u2014 move on to the next explainer!" if total and passed == total
      else "Rerun each check cell as you fill in the TODOs above.")
